### Reminder

- On what data the model will work better
- Normalize distributions of parameters that feed the model
- Gap between manufactured and sold as a predictor

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

In [3]:
df = pd.read_csv('datasets/car_prices/archive(2)/car_prices.csv')
df.head()

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,kia motors america inc,20500.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,kia motors america inc,20800.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,45.0,1331.0,gray,black,financial services remarketing (lease),31900.0,30000.0,Thu Jan 15 2015 04:30:00 GMT-0800 (PST)
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,41.0,14282.0,white,black,volvo na rep/world omni,27500.0,27750.0,Thu Jan 29 2015 04:30:00 GMT-0800 (PST)
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,43.0,2641.0,gray,black,financial services remarketing (lease),66000.0,67000.0,Thu Dec 18 2014 12:30:00 GMT-0800 (PST)


### Feature Engineering

In [4]:
# ''' kto pridumal peresozdovat novie imena odnogo i togozhe'''

# def normalize_body(val):
#     if pd.isna(val):
#         return np.nan
#     v = val.lower()

#     if 'cab' in v:
#         return 'cab'
#     if 'sedan' in v:
#         return 'sedan'
#     if 'koup' in v or 'coupe' in v:
#         return 'coupe'
#     if 'hatch' in v:
#         return 'hatchback'
#     if 'mini' in v:
#         return 'minivan'
#     if 'convert' in v:
#         return 'convertible'
#     if 'wagon' in v:
#         return 'wagon'
#     if 'van' in v:
#         return 'van'
#     if 'suv' in v:
#         return 'suv'

#     return 'other'
# df['body']=df['body'].apply(normalize_body)

# import re

# def normalize_make(s: str):
#     if pd.isna(s):
#         return s
    
#     s = s.lower().strip()

#     # базовая очистка
#     s = re.sub(r'\b(tk|truck)\b', '', s)
#     s = re.sub(r'\s+', ' ', s).strip()

#     # алиасы брендов
#     alias_map = {
#         'vw': 'volkswagen',
#         'landrover': 'land rover',
#         'mercedes': 'mercedes-benz',
#         'mercedes-b': 'mercedes-benz',
#         'gmc truck': 'gmc',
#         'ford tk': 'ford',
#         'chev': 'chevrolet'
#     }

#     return alias_map.get(s, s)
# df['make'] = df['make'].apply(normalize_make)

In [4]:
df.head()

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,kia motors america inc,20500.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,kia motors america inc,20800.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,45.0,1331.0,gray,black,financial services remarketing (lease),31900.0,30000.0,Thu Jan 15 2015 04:30:00 GMT-0800 (PST)
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,41.0,14282.0,white,black,volvo na rep/world omni,27500.0,27750.0,Thu Jan 29 2015 04:30:00 GMT-0800 (PST)
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,43.0,2641.0,gray,black,financial services remarketing (lease),66000.0,67000.0,Thu Dec 18 2014 12:30:00 GMT-0800 (PST)


In [5]:
df.loc[:,'make']=df.loc[:,'make'].fillna(df.make.mode()[0])
df.loc[:,'model']=df.loc[:,'model'].fillna(df.model.mode()[0])
df.loc[:,'body']=df.loc[:,'body'].fillna(df.body.mode()[0])
df.loc[:,'trim']=df.loc[:,'trim'].fillna(df.trim.mode()[0])
df.loc[:,'interior']=df.loc[:,'interior'].fillna(df.interior.mode()[0])
df.loc[:,'color']=df.loc[:,'color'].fillna(df.color.mode()[0])
df.loc[:,'transmission']=df.loc[:,'transmission'].fillna(df.transmission.mode()[0])
df.loc[:,'condition']=df.loc[:,'condition'].fillna(df.condition.mean())
df.loc[:,'odometer']=df.loc[:,'odometer'].fillna(df.odometer.mean())
df.loc[:,'mmr']=df.loc[:,'mmr'].fillna(df.mmr.mean())

In [6]:
df['saledate'] = pd.to_datetime(
    df['saledate'].astype(str),
    errors='coerce',
    utc=True
)
df['saledate'].isna().sum()

/tmp/ipykernel_2455/1564496658.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['saledate'] = pd.to_datetime(


38

In [7]:
df['saledate'] = pd.to_datetime(df['saledate'], utc=True)
df['sale_year'] = df['saledate'].dt.year
df['car_age'] = df['sale_year'] - df['year']

    Mileage, Prices  distribution are heavily right skewed, but that is normal. High mileage and high-end cars much less often than ordinary cars. So I will try to not cut outliers.
- Mileage between 1 - 1,000,000mi
- Selling price between 1 - 230,000 dollars
- Also drop 38 missed values from sellingprice and saledate

In [8]:
df = df.dropna()

In [9]:
X= df.drop('sellingprice', axis=1)
y= df['sellingprice']

In [10]:
df.columns

Index(['year', 'make', 'model', 'trim', 'body', 'transmission', 'vin', 'state',
       'condition', 'odometer', 'color', 'interior', 'seller', 'mmr',
       'sellingprice', 'saledate', 'sale_year', 'car_age'],
      dtype='object')

### With outliers

In [13]:
X = X.drop(['mmr','trim','model','vin','seller','saledate', 'sale_year', 'year'], axis=1)
Xtr, Xte, ytr, yte = train_test_split(X, y, train_size=0.8, random_state=42)

In [14]:
X.head()

,make,body,transmission,state,condition,odometer,color,interior,car_age
0,Kia,SUV,automatic,ca,5.0,16639.0,white,black,-1.0
1,Kia,SUV,automatic,ca,5.0,9393.0,white,beige,-1.0
2,BMW,Sedan,automatic,ca,45.0,1331.0,gray,black,1.0
3,Volvo,Sedan,automatic,ca,41.0,14282.0,white,black,0.0
4,BMW,Sedan,automatic,ca,43.0,2641.0,gray,black,0.0


In [15]:
cat_features = ['make','body', 'transmission','state', 'color', 'interior']
cat_mod = CatBoostRegressor(
    iterations=1000,
    depth=9,
    learning_rate=0.01,
    loss_function='RMSE',
    verbose=100
)
cat_mod.fit(Xtr, ytr, cat_features=cat_features)
predd = cat_mod.predict(Xte)
mse = mean_squared_error(yte, predd)
np.sqrt(mse), cat_mod.score(Xte,yte)

0:	learn: 9687.3416191	total: 247ms	remaining: 4m 6s
100:	learn: 6225.7772606	total: 12.6s	remaining: 1m 52s
200:	learn: 5227.8177435	total: 25.3s	remaining: 1m 40s
300:	learn: 4870.7800261	total: 38.6s	remaining: 1m 29s
400:	learn: 4705.6590025	total: 53.9s	remaining: 1m 20s
500:	learn: 4603.1042556	total: 1m 8s	remaining: 1m 8s
600:	learn: 4519.4358519	total: 1m 23s	remaining: 55.1s
700:	learn: 4450.9140364	total: 1m 37s	remaining: 41.6s
800:	learn: 4396.0048989	total: 1m 52s	remaining: 27.9s
900:	learn: 4347.7725256	total: 2m 7s	remaining: 14s
999:	learn: 4310.2632610	total: 2m 21s	remaining: 0us


(4400.58863532191, 0.7962496757617284)

In [16]:
fi = cat_mod.get_feature_importance(prettified=True)
fi

,Feature Id,Importances
0,make,39.469551
1,body,21.930377
2,car_age,15.192250
3,odometer,13.729606
4,condition,5.222997
5,interior,2.688206
6,state,1.126877
7,color,0.478969
8,transmission,0.161166


In [17]:
X_corrected = X.drop(['interior','state','color','transmission'], axis=1)
Xtrc, Xtec, ytrc, ytec = train_test_split(X_corrected, y, train_size=0.8, random_state=42)
cat_features = ['make','body']

cat_mod_corrected = CatBoostRegressor(
    iterations=1000,
    depth=9,
    learning_rate=0.01,
    loss_function='RMSE',
    verbose=100
)
cat_mod_corrected.fit(Xtrc, ytrc, cat_features=cat_features)
preddc = cat_mod_corrected.predict(Xtec)
msec = mean_squared_error(ytec, preddc)
np.sqrt(msec), cat_mod_corrected.score(Xtec,ytec)

0:	learn: 9694.0274718	total: 115ms	remaining: 1m 55s
100:	learn: 6288.7054468	total: 7.83s	remaining: 1m 9s
200:	learn: 5342.2884775	total: 15.3s	remaining: 1m
300:	learn: 4989.2605834	total: 22.8s	remaining: 53s
400:	learn: 4814.7325302	total: 30s	remaining: 44.9s
500:	learn: 4700.5975863	total: 37.1s	remaining: 36.9s
600:	learn: 4619.0155879	total: 44.3s	remaining: 29.4s
700:	learn: 4551.7083364	total: 51.5s	remaining: 22s
800:	learn: 4502.3182655	total: 58.8s	remaining: 14.6s
900:	learn: 4458.9313111	total: 1m 6s	remaining: 7.25s
999:	learn: 4423.0003513	total: 1m 13s	remaining: 0us


(4490.7899811972375, 0.7878112995385894)

In [18]:
X_ohe = pd.get_dummies(X_corrected, columns=['make','body'], drop_first=False)
Xtr_ohe, Xte_ohe, ytr_ohe, yte_ohe = train_test_split(X_ohe, y, train_size=0.8, random_state=42)
X_ohe.head()

,condition,odometer,car_age,make_Acura,make_Aston Martin,make_Audi,make_BMW,make_Bentley,make_Buick,make_Cadillac,...,body_regular-cab,body_sedan,body_supercab,body_supercrew,body_suv,body_transit van,body_tsx sport wagon,body_van,body_wagon,body_xtracab
0,5.0,16639.0,-1.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,5.0,9393.0,-1.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,45.0,1331.0,1.0,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,41.0,14282.0,0.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,43.0,2641.0,0.0,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [19]:
mod_xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=9,
    min_child_weight=5,
    gamma=1,
    subsample=0.7,
    colsample_bytree=0.7
)
mod_xgb.fit(Xtr_ohe,ytr_ohe)
predd_xgb = mod_xgb.predict(Xte_ohe)
np.sqrt(mean_squared_error(yte_ohe, predd_xgb))

4384.615814300587

In [20]:
mod_xgb.score(Xte_ohe,yte_ohe)

0.7977260969223431

### Without outliers

In [21]:
mask = pd.Series(True, index=df.index)
for col in ['odometer','mmr','sellingprice']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3-q1
    low = q1 - iqr*1.5
    high =q3 + iqr*1.5
    mask &= (df[col]>=low)&(df[col]<=high)
print(f"\033[34mNumber of outliers from mileage, mmr, price\033[0m: {len(df)-mask.sum()}, {((len(df)-mask.sum())/len(df))*100:.2f}")
df_outless = df[mask].copy()

Number of outliers from mileage, mmr, price: 28319, 5.07


In [22]:
y_outless= df['sellingprice']
X_outless = df.drop(['sellingprice','mmr','trim','model','vin','seller','saledate', 'sale_year', 'year',], axis=1)
X_ohe_out = pd.get_dummies(X_outless, columns=['make','body','transmission','state','color','interior'], drop_first=False)
Xtr_out, Xte_out, ytr_out, yte_out = train_test_split(X_ohe_out, y_outless, train_size=0.8, random_state=42)

In [23]:
mod_xgb_out = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=9,
    min_child_weight=5,
    gamma=1,
    subsample=0.7,
    colsample_bytree=0.7
)
mod_xgb_out.fit(Xtr_out,ytr_out)
predd_xgb_out = mod_xgb_out.predict(Xte_out)
np.sqrt(mean_squared_error(yte_out, predd_xgb_out))

4296.265761089402

In [24]:
mod_xgb_out.score(Xte_out,yte_out)

0.8057956114884274

### Simple nn
Performed not so well. Further deepening can lead to overfitting

In [46]:
print('ebadi: ', torch.cuda.is_available())
print('dranduliator: ', torch.cuda.get_device_name(0))

ebadi:  True
dranduliator:  NVIDIA GeForce GTX 1050


In [47]:
scaler = StandardScaler()
Xtr_out_sc = scaler.fit_transform(Xtr_out)
Xte_out_sc = scaler.transform(Xte_out)

ytr_np = ytr_out.values.astype('float32')
yte_np = yte_out.values.astype('float32')

tr_ds = TensorDataset(torch.from_numpy(Xtr_out_sc).float(),
                      torch.from_numpy(ytr_np).float())

te_ds = TensorDataset(torch.from_numpy(Xte_out_sc).float(),
                      torch.from_numpy(yte_np).float())

tr_load = DataLoader(tr_ds, batch_size=64, shuffle=True)
te_load = DataLoader(te_ds, batch_size=64, shuffle=False)

In [48]:
device = torch.device('cuda')

class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(262, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            
            nn.Linear(512, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),

            nn.Linear(512, 256),nn.BatchNorm1d(256),nn.ReLU(),nn.Dropout(0.3),

            nn.Linear(256, 128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(0.25),

            nn.Linear(128, 64),nn.ReLU(),

            nn.Linear(64, 32),nn.ReLU(),

            nn.Linear(32, 16),nn.ReLU(),

            nn.Linear(16, 1),            
        )
    def forward(self, x):
        return self.fc(x)

In [49]:
mlp_mod = MLP().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(mlp_mod.parameters(), lr=10**-3, weight_decay=10**-4)
epochs = 10

In [50]:
def tr_ep(mod, load):
    mod.train()
    total_loss = 0

    for X,y in load:
        X = X.to(device).float()
        y = y.to(device).float().view(-1,1)
        optimizer.zero_grad()
        preds = mod(X)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
    return total_loss/len(load.dataset)

def te_ep(mod, load):
    mod.eval()
    total_loss=0
    with torch.no_grad():
        for X,y in load:
            X = X.to(device).float()
            y = y.to(device).float().view(-1,1)
            preds = mod(X)
            loss = criterion(preds, y)
            total_loss += loss.item() * X.size(0)
        return total_loss/len(load.dataset)


for ep in range(epochs):
    tr_loss = tr_ep(mlp_mod, tr_load)
    te_loss = te_ep(mlp_mod, te_load)
    
    print(f"eps: {ep},   training loss: {np.sqrt(tr_loss):.3f},    validation loss: {np.sqrt(te_loss):.3f}") 

eps: 0,   training loss: 5629.384,    validation loss: 4724.511
eps: 1,   training loss: 4814.604,    validation loss: 4823.511
eps: 2,   training loss: 4702.250,    validation loss: 4386.114
eps: 3,   training loss: 4635.849,    validation loss: 4398.319
eps: 4,   training loss: 4588.970,    validation loss: 4284.415
eps: 5,   training loss: 4540.257,    validation loss: 4407.852
eps: 6,   training loss: 4486.870,    validation loss: 4282.760
eps: 7,   training loss: 4439.791,    validation loss: 4451.375
eps: 8,   training loss: 4401.089,    validation loss: 4277.880
eps: 9,   training loss: 4370.857,    validation loss: 4221.907


In [51]:
te_loss**0.5

4221.906587541919